# 01 – Kaggle EDA and Feature Analysis

This notebook explores the `Store Item Demand Forecasting Challenge` dataset and
the synthetic/demo series used when Kaggle data is not available.

It is designed to run in **two modes**:

1. **Kaggle mode** – when `data/raw/train.csv` exists.
2. **Demo mode** – when Kaggle data is missing; falls back to a synthetic series
   generated by the `demandcast.data.synthetic` module or the small demo CSV
   under `assets/demo/demo_series.csv`.


In [ ]:
from pathlib import Path

import pandas as pd

import sys
PROJECT_ROOT = Path.cwd().resolve().parents[0]
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from demandcast.data import (
    build_target_series,
    generate_demo_series,
    load_kaggle_train_df,
    locate_kaggle_train_csv,
)
from demandcast.features import (
    FeatureConfig,
    build_time_series_features,
)
from demandcast.evaluation.metrics import rmse, mae, mape, smape, wape
from demandcast.logging_utils import get_logger
from demandcast.paths import DEMO_DATA_DIR

logger = get_logger(__name__)


## Load Kaggle data or fall back to demo series

This cell tries to load the Kaggle competition data. If it is not found, it
falls back to the synthetic demo series (or the committed demo CSV).

In [ ]:
try:
    csv_path = locate_kaggle_train_csv()
    raw_df = load_kaggle_train_df(csv_path)
    source = f"Kaggle CSV – {csv_path}"
except FileNotFoundError:
    demo_csv = DEMO_DATA_DIR / "demo_series.csv"
    if demo_csv.exists():
        demo_df = pd.read_csv(demo_csv)
        demo_df["date"] = pd.to_datetime(demo_df["ds"])
        demo_df["store"] = demo_df.get("store", 1)
        demo_df["item"] = demo_df.get("item", 1)
        demo_df["sales"] = demo_df.get("y", demo_df.get("sales", 1.0))
        raw_df = demo_df[["date", "store", "item", "sales"]].copy()
        source = f"Demo CSV – {demo_csv}"
    else:
        raw_df = generate_demo_series()
        raw_df["date"] = raw_df["ds"]
        raw_df["store"] = raw_df.get("store", 1)
        raw_df["item"] = raw_df.get("item", 1)
        raw_df["sales"] = raw_df.get("y", 1.0)
        raw_df = raw_df[["date", "store", "item", "sales"]].copy()
        source = "Synthetic demo series (generated on the fly)"

logger.info("Using data source: %s", source)
raw_df.head()

## Build a univariate target series

We collapse the store–item panel into a single series for a default
`(store, item)` pair or aggregate as needed.

In [ ]:
series_df = build_target_series(raw_df)
series_df.head()

## Feature engineering

We apply the time-series feature pipeline (calendar features, lags, and rolling
statistics) to construct a supervised learning dataset for downstream models
such as LightGBM.

In [ ]:
features_cfg = FeatureConfig()
features_df = build_time_series_features(series_df, config=features_cfg)
features_df.head()

## Quick baseline metrics

To get a rough sense of difficulty, we can compute very simple baselines such
as **naïve lag-1** or **moving average** and evaluate them with RMSE, MAE,
MAPE, sMAPE, and WAPE. For brevity, the exact code is omitted here; the
core implementation lives in `src/demandcast/evaluation/metrics.py` and the
scripts in `scripts/` provide repeatable CLI entry points.